Каждый из нас писал в школе и университете изложения, сочинения, рефераты. А значит, в каждом из нас живет великий русский писатель.
В этой работе будем раскрывать свои таланты, находить себя в ряду таких гениев, как Пушкин, Гоголь, Грибоедов

В этой работе
- скачаем корпус текстов 20-ми русских писателей. Каждый текст разобьем на обучающую и тестовую выборки.
- разработаем и обучим нейронную сеть определяющию авторство фрагментов текста (по тестовой выборке)
- скачаем СВОЕ сочинение (или чье-нибудь - есть в архиве). Сделаем из него проверочную выборку
- предложим нейронке предсказать автора сочинения (по проверочной выборке)
- объявим себя великим писателем, например, Гончаровым

Ссылка на архив: https://storage.yandexcloud.net/aiueducation/Content/base/l7/20writers.zip

В работе рекомендуется пользоваться материалами из ноутбука практического занятия "Рекуррентные и одномерные сверточные нейронные сети". Допускается выбрать лучший вариант нейронки и адаптировать ее структуру, параметры обучения и формирования датасетов под свои *нужды*

## 1. Импорт библиотек

In [1]:
import numpy as np
import os
import re
import time
import gdown
import matplotlib.pyplot as plt

from tensorflow.keras import utils
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense, Dropout, Embedding,
    LSTM, Bidirectional, Conv1D,
    MaxPooling1D, GlobalMaxPooling1D,
    SpatialDropout1D
)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

%matplotlib inline

## 2. Скачивание датасета

In [2]:
gdown.download(
    'https://storage.yandexcloud.net/aiueducation/Content/base/l7/20writers.zip',
    None,
    quiet=True
)

'20writers.zip'

## 3. Распаковка архива


In [3]:
!unzip -o 20writers.zip

Archive:  20writers.zip
  inflating: Грибоедов.txt  
  inflating: Достоевский.txt  
  inflating: Каверин.txt      
  inflating: Катаев.txt        
  inflating: Куприн.txt        
  inflating: Лермонтов.txt  
  inflating: Лесков.txt        
  inflating: Носов.txt          
  inflating: Пастернак.txt  
  inflating: Пушкин.txt        
  inflating: Толстой.txt      
  inflating: Тургенев.txt    
  inflating: Чехов.txt          
  inflating: Шолохов.txt      
  inflating: Беляев.txt        
  inflating: Булгаков.txt    
  inflating: Васильев.txt    
  inflating: Гоголь.txt        
  inflating: Гончаров.txt    
  inflating: Горький.txt      


## 4. Определение пути

In [8]:
FILE_DIR = '.'

## 5. Загрузка текстов и создание списка классов

In [9]:
CLASS_LIST = []
texts = []
labels = []

file_list = os.listdir(FILE_DIR)

for file_name in file_list:

    if file_name.endswith('.txt'):

        class_name = file_name.replace('.txt', '')

        CLASS_LIST.append(class_name)

        with open(file_name, 'r', encoding='utf-8') as f:
            text = f.read().replace('\n', ' ')

        texts.append(text)

print("Классы:", CLASS_LIST)

Классы: ['Лесков', 'Носов', 'Тургенев', 'Грибоедов', 'Катаев', 'Достоевский', 'Гоголь', 'Каверин', 'Лермонтов', 'Беляев', 'Чехов', 'Пастернак', 'Пушкин', 'Горький', 'Гончаров', 'Шолохов', 'Куприн', 'Булгаков', 'Васильев', 'Толстой']


## 6. Определение гиперпараметров

In [16]:
VOCAB_SIZE = 20000
WIN_SIZE = 40
WIN_STEP = 5

EPOCHS = 10
BATCH_SIZE = 512

## 7. Создание токенизатора и преобразование текстов в последовательности

In [17]:
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='UNK')
tokenizer.fit_on_texts(texts)

seqs = tokenizer.texts_to_sequences(texts)

## 8. Функция create_dataset для нарезки окон

In [18]:
def create_dataset(seqs, win_size, step):

    X = []
    y = []

    for class_id, seq in enumerate(seqs):

        for i in range(0, len(seq) - win_size, step):

            X.append(seq[i:i+win_size])
            y.append(class_id)

    return np.array(X), utils.to_categorical(y, len(CLASS_LIST))

## 9. Создание выборок и разделение на train/test

In [19]:
X, y = create_dataset(seqs, WIN_SIZE, WIN_STEP)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape, X_test.shape)

(1438900, 40) (359726, 40)


## 10. Создание модели

In [20]:
model = Sequential()

model.add(Embedding(VOCAB_SIZE, 128, input_length=WIN_SIZE))
model.add(SpatialDropout1D(0.3))

model.add(Bidirectional(LSTM(64, return_sequences=True)))
model.add(GlobalMaxPooling1D())

model.add(Dense(128, activation='relu'))
model.add(Dropout(0.3))

model.add(Dense(len(CLASS_LIST), activation='softmax'))

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d_1             │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_1          │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

## 11. Обучение модели

In [21]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)

Epoch 1/10
2811/2811 ━━━━━━━━━━━━━━━━━━━━ 64s 22ms/step - accuracy: 0.6761 - loss: 1.0613 - val_accuracy: 0.8200 - val_loss: 0.5762
Epoch 2/10
2811/2811 ━━━━━━━━━━━━━━━━━━━━ 62s 22ms/step - accuracy: 0.8281 - loss: 0.5512 - val_accuracy: 0.8699 - val_loss: 0.4098
Epoch 3/10
2811/2811 ━━━━━━━━━━━━━━━━━━━━ 61s 22ms/step - accuracy: 0.8642 - loss: 0.4280 - val_accuracy: 0.8904 - val_loss: 0.3392
Epoch 4/10
2811/2811 ━━━━━━━━━━━━━━━━━━━━ 62s 22ms/step - accuracy: 0.8855 - loss: 0.3551 - val_accuracy: 0.9092 - val_loss: 0.2802
Epoch 5/10
2811/2811 ━━━━━━━━━━━━━━━━━━━━ 61s 22ms/step - accuracy: 0.9002 - loss: 0.3060 - val_accuracy: 0.9207 - val_loss: 0.2413
Epoch 6/10
2811/2811 ━━━━━━━━━━━━━━━━━━━━ 61s 22ms/step - accuracy: 0.9115 - loss: 0.2691 - val_accuracy: 0.9297 - val_loss: 0.2125
Epoch 7/10
2811/2811 ━━━━━━━━━━━━━━━━━━━━ 61s 22ms/step - accuracy: 0.9196 - loss: 0.2409 - val_accuracy: 0.9368 - val_loss: 0.1894
Epoch 8/10
2811/2811 ━━━━━━━━━━━━━━━━━━━━ 61s 22ms/step - accuracy: 0.9264 -

## 12. Предсказание меток для тестовой выборки

In [26]:
pred = model.predict(X_test)

y_pred = np.argmax(pred, axis=1)
y_true = np.argmax(y_test, axis=1)

11242/11242 ━━━━━━━━━━━━━━━━━━━━ 49s 4ms/step


## 13. Вывод первых 10 предсказаний

In [27]:
for i in range(10):

    print("Истинный автор:", CLASS_LIST[y_true[i]])
    print("Предсказание  :", CLASS_LIST[y_pred[i]])
    print("-" * 40)

Истинный автор: Достоевский
Предсказание  : Достоевский
----------------------------------------
Истинный автор: Гоголь
Предсказание  : Гоголь
----------------------------------------
Истинный автор: Горький
Предсказание  : Горький
----------------------------------------
Истинный автор: Толстой
Предсказание  : Толстой
----------------------------------------
Истинный автор: Пастернак
Предсказание  : Пастернак
----------------------------------------
Истинный автор: Горький
Предсказание  : Горький
----------------------------------------
Истинный автор: Носов
Предсказание  : Носов
----------------------------------------
Истинный автор: Чехов
Предсказание  : Чехов
----------------------------------------
Истинный автор: Достоевский
Предсказание  : Достоевский
----------------------------------------
Истинный автор: Катаев
Предсказание  : Катаев
----------------------------------------


## 14. Оценка точности на тестовой выборке

In [29]:
loss, acc = model.evaluate(X_test, y_test)
print("Test accuracy:", acc)

11242/11242 ━━━━━━━━━━━━━━━━━━━━ 85s 8ms/step - accuracy: 0.9515 - loss: 0.1458
Test accuracy: 0.9515464305877686


## 15. Создание текста для проверки

In [47]:
my_text = """
Иногда человек сам не понимает, почему поступает так, а не иначе.
Он мучается, сомневается, ищет оправдания своим мыслям и действиям,
и в этом бесконечном внутреннем диалоге теряет ощущение ясности.

Каждое решение кажется ему одновременно правильным и ошибочным,
и от этого душа становится тяжелее, чем прежде.

Но, несмотря на это, человек продолжает жить,
потому что остановиться — значит окончательно признать своё бессилие
перед самим собой и перед жизнью.
"""

## 16. Предсказание автора для текста

In [48]:
seq = tokenizer.texts_to_sequences([my_text])
seq = pad_sequences(seq, maxlen=WIN_SIZE)

pred = model.predict(seq)[0]

print("Автор текста:", CLASS_LIST[np.argmax(pred)])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step
Автор текста: Васильев
